In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("usarrests.csv", index_col=0)
print(df.shape)
df.head()

(50, 3)


,Murder,Assault,UrbanPop
Alabama,13.2,236.0,58
Alaska,10.0,263.0,48
Arizona,8.1,294.0,80
Arkansas,8.8,190.0,50
California,9.0,276.0,91


Summary table

In [2]:
df.isna().sum()

Murder      0
Assault     1
UrbanPop    0
dtype: int64

In [3]:
df.describe().round(2)

,Murder,Assault,UrbanPop
count,50.00,49.00,50.00
mean,7.79,182.18,74.20
std,4.36,130.88,73.41
min,0.80,45.00,6.00
25%,4.08,109.00,53.25
50%,7.25,159.00,66.00
75%,11.25,249.00,77.75
max,17.40,879.00,570.00


The summary shows two problems. The maximum UrbanPop is 570, but a percentage cannot be above 100. The maximum Assault is 879, which is far above the 75% value of 249. Assault also has a count of 49, so one value is missing.

 Flagged values

In [4]:
df[df.isna().any(axis=1)]

,Murder,Assault,UrbanPop
Georgia,17.4,NaN,60


In [5]:
df[(df["UrbanPop"] > 100) | (df["UrbanPop"] < 20)]

,Murder,Assault,UrbanPop
Iowa,2.2,56.0,570
New York,11.1,254.0,6


In [6]:
df[df["Assault"] > 400]

,Murder,Assault,UrbanPop
South Carolina,14.4,879.0,48


Iowa, UrbanPop = 570: impossible. A percentage cannot be above 100, so this has to be a typo. I set it to missing and filled it with the median.

New York, UrbanPop = 6: extreme. It is a valid percentage, but New York is a very urban state and every other state is 32 or higher, so it is probably a typo. I set it to missing and filled it with the median.

South Carolina, Assault = 879: extreme. A high arrest rate is possible, but it is much higher than the next highest value (337), so it is probably a typo. I set it to missing and filled it with the median.

Georgia, Assault = blank: missing. The value was not recorded. I filled it with the median so I could keep the row.

Fix the values

In [7]:
clean = df.copy()
clean.loc["Iowa", "UrbanPop"] = np.nan
clean.loc["New York", "UrbanPop"] = np.nan
clean.loc["South Carolina", "Assault"] = np.nan

fill_values = clean.median()
print(fill_values)

clean = clean.fillna(fill_values)
print(clean.isna().sum())

Murder        7.25
Assault     157.50
UrbanPop     66.00
dtype: float64
Murder      0
Assault     0
UrbanPop    0
dtype: int64


In [8]:
clean.describe().round(2)

,Murder,Assault,UrbanPop
count,50.00,50.00,50.00
mean,7.79,167.26,65.32
std,4.36,81.66,14.12
min,0.80,45.00,32.00
25%,4.08,109.00,54.50
50%,7.25,157.50,66.00
75%,11.25,246.25,76.50
max,17.40,337.00,91.00


Cleaning Log

1. Loaded 50 rows and 3 columns.
2. Iowa UrbanPop was 570 (impossible). Replaced with the median, 66.0.
3. New York UrbanPop was 6 (extreme). Replaced with the median, 66.0.
4. South Carolina Assault was 879 (extreme). Replaced with the median, 157.5.
5. Georgia Assault was blank. Filled with the median, 157.5.
6. Result: 4 values fixed, 0 missing values, shape still (50, 3).

If these values had stayed in the data, the average urban population would look too high and one state would look much more dangerous than it is.

Repo URL: https://github.com/harveyvasquez/cs82a-portfolio